In [10]:
import pandas as pd
import numpy as np

RAW_PATH = "../data/merged/multi_class_dataset_clean.csv"
OUT_PATH = "../data/merged/multi_class_dataset_clean_intermediate.csv"

df = pd.read_csv(RAW_PATH)
print("Raw shape:", df.shape)

# Appearance impedance columns
Z_cols = ["R1-PA:Z", "R2-PA:Z", "R3-PA:Z", "R4-PA:Z"]

# ---- 1. Add binary infinity flag ----
for col in Z_cols:
    df[col + "_inf_flag"] = np.isinf(df[col]).astype(int)

# ---- 2. Replace inf with NaN ----
df[Z_cols] = df[Z_cols].replace([np.inf, -np.inf], np.nan)

# ---- 3. Fill NaN in Z only (median preserves physics) ----
df[Z_cols] = df[Z_cols].fillna(df[Z_cols].median())

# No other cleaning, no capping, no scaling here.
df.to_csv(OUT_PATH, index=False)
print("Saved CLEANED dataset:", OUT_PATH)


Raw shape: (78377, 129)
Saved CLEANED dataset: ../data/merged/multi_class_dataset_clean_intermediate.csv


In [11]:
import pandas as pd
import os

OUTPUT_DIR = "../data/datasets_hierarchical"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Saving all datasets to:", OUTPUT_DIR)

# Scenario groups
faults = list(range(1, 7))
maintenance = [13, 14]
normal = [41]

data_injection = list(range(7, 13))
remote_tripping = list(range(15, 21))
relay_setting = list(range(21, 31)) + list(range(35, 41))

all_attacks = data_injection + remote_tripping + relay_setting
all_non_attacks = faults + maintenance + normal

df = pd.read_csv("../data/merged/multi_class_dataset_clean_intermediate.csv")
print("Loaded dataset shape:", df.shape)

# ------------------------------------------------------------
# M1 – Attack vs Non-Attack
# ------------------------------------------------------------
df_m1 = df.copy()
df_m1["label"] = df_m1["marker"].apply(lambda s: 1 if s in all_attacks else 0)
df_m1["label_name"] = df_m1["label"].map({0: "Non-Attack", 1: "Attack"})
df_m1.to_csv(f"{OUTPUT_DIR}/M1_Attack_vs_NonAttack.csv", index=False)

# ------------------------------------------------------------
# M2 – Natural 3-Class
# ------------------------------------------------------------
df_m2 = df[df["marker"].isin(all_non_attacks)].copy()

def map_natural(s):
    if s in faults: return 0
    elif s in maintenance: return 1
    return 2

df_m2["label"] = df_m2["marker"].apply(map_natural)
df_m2["label_name"] = df_m2["label"].map({
    0: "Short-Circuit Fault",
    1: "Line Maintenance",
    2: "Normal Operation"
})
df_m2.to_csv(f"{OUTPUT_DIR}/M2_Natural_3Class.csv", index=False)

# ------------------------------------------------------------
# M3 – Attack 3-Class
# ------------------------------------------------------------
df_m3 = df[df["marker"].isin(all_attacks)].copy()

def map_attack(s):
    if s in data_injection: return 0
    elif s in remote_tripping: return 1
    return 2

df_m3["label"] = df_m3["marker"].apply(map_attack)
df_m3["label_name"] = df_m3["label"].map({
    0: "Data Injection",
    1: "Remote Tripping",
    2: "Relay Setting Change"
})
df_m3.to_csv(f"{OUTPUT_DIR}/M3_Attack_3Class.csv", index=False)

# ------------------------------------------------------------
# M4 – Data Injection Internal
# ------------------------------------------------------------
df_m4 = df[df["marker"].isin(data_injection)].copy()
df_m4["label"] = df_m4["marker"]
df_m4.to_csv(f"{OUTPUT_DIR}/M4_DataInjection_Internal.csv", index=False)

# ------------------------------------------------------------
# M5 – Remote Tripping Internal
# ------------------------------------------------------------
df_m5 = df[df["marker"].isin(remote_tripping)].copy()
df_m5["label"] = df_m5["marker"]
df_m5.to_csv(f"{OUTPUT_DIR}/M5_RemoteTripping_Internal.csv", index=False)

# ------------------------------------------------------------
# M6 – Relay Setting Internal
# ------------------------------------------------------------
df_m6 = df[df["marker"].isin(relay_setting)].copy()
df_m6["label"] = df_m6["marker"]
df_m6.to_csv(f"{OUTPUT_DIR}/M6_RelaySetting_Internal.csv", index=False)

print("\n=== All hierarchical datasets created successfully! ===")


Saving all datasets to: ../data/datasets_hierarchical
Loaded dataset shape: (78377, 133)

=== All hierarchical datasets created successfully! ===


In [4]:
DATA_DIR = "../data/datasets_hierarchical/cleaned"
print("Loading cleaned datasets from:", DATA_DIR)
# -----------------------------------------------------------
# EXPECTED CLEANED DATASET FILENAMES
# -----------------------------------------------------------
datasets = {
    "M1_Attack_vs_NonAttack": "M1_Attack_vs_NonAttack_clean.csv",
    "M2_Natural_3Class": "M2_Natural_3Class_clean.csv",
    "M3_Attack_3Class": "M3_Attack_3Class_clean.csv",
    "M4_DataInjection_Internal": "M4_DataInjection_Internal_clean.csv",
    "M5_RemoteTripping_Internal": "M5_RemoteTripping_Internal_clean.csv",
    "M6_RelaySetting_Internal": "M6_RelaySetting_Internal_clean.csv"
}

cleaned_datasets = {}

# -----------------------------------------------------------
# LOAD ALL DATASETS
# -----------------------------------------------------------
print("\n==================== DATASET LOADING ====================\n")

for name, filename in datasets.items():
    path = os.path.join(DATA_DIR, filename)

    if not os.path.exists(path):
        print(f"❌ ERROR: File not found → {filename}")
        print("   Please check the cleaned folder.")
        continue

    df = pd.read_csv(path)
    cleaned_datasets[name] = df

    print(f"✔ Loaded {name}: {df.shape[0]} rows × {df.shape[1]} columns")
    X = df.drop(["label", "label_name"], axis=1)  # model inputs
    y = df["label"]                               # numeric labels (OK)


print("\n==========================================================\n")


Loading cleaned datasets from: ../data/datasets_hierarchical/cleaned

==================== DATASET LOADING ====================

✔ Loaded M1_Attack_vs_NonAttack: 78377 rows × 131 columns
✔ Loaded M2_Natural_3Class: 22714 rows × 131 columns
✔ Loaded M3_Attack_3Class: 55663 rows × 131 columns
✔ Loaded M4_DataInjection_Internal: 9582 rows × 130 columns


KeyError: "['label_name'] not found in axis"

In [12]:
df_m1 = pd.read_csv("../data/datasets_hierarchical/M1_Attack_vs_NonAttack.csv")
top10_m1 = eda_basic(df_m1)


1. DATASET OVERVIEW
Shape: (78377, 131)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78377 entries, 0 to 78376
Columns: 131 entries, R1-PA1:VH to label_name
dtypes: float64(128), int64(2), object(1)
memory usage: 78.3+ MB
None

Summary statistics:


/Users/cherylchau/Library/Mobile Documents/com~apple~CloudDocs/FYP-data/myenv/lib/python3.9/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/cherylchau/Library/Mobile Documents/com~apple~CloudDocs/FYP-data/myenv/lib/python3.9/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/cherylchau/Library/Mobile Documents/com~apple~CloudDocs/FYP-data/myenv/lib/python3.9/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/cherylchau/Library/Mobile Documents/com~apple~CloudDocs/FYP-data/myenv/lib/python3.9/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


             count           mean          std            min            25%  \
R1-PA1:VH  78377.0     -15.802424   100.876750    -179.988962    -100.416583   
R1-PM1:V   78377.0  131670.634209  1039.382656  129365.536650  131057.982300   
R1-PA2:VH  78377.0       2.175196   111.743169    -179.994691    -102.129727   
R1-PM2:V   78377.0  131361.012902  1070.855517  129001.974200  130732.029800   
R1-PA3:VH  78377.0       6.834315    97.065063    -179.994691     -69.459674   
R1-PM3:V   78377.0  131736.200705  1038.943651  129440.756300  131133.202100   
R1-PA4:IH  78377.0     -14.334996    99.601107    -179.994691     -98.159129   
R1-PM4:I   78377.0     380.592779   119.435979      79.469740     305.793700   
R1-PA5:IH  78377.0       3.538540   109.504977    -179.994691     -94.790138   
R1-PM5:I   78377.0     383.504558   115.162266      89.083015     311.836330   

                     50%            75%            max  
R1-PA1:VH     -28.865614      68.096034     179.994691  
R1-PM

NameError: name 'plt' is not defined